In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
from datasets import load_dataset

# 1. Defina o caminho exato do arquivo (ajuste se estiver em uma pasta específica)
path_to_jsonl = "/content/drive/MyDrive/dataset_finetuning_postech.jsonl"

# 2. Carregue o dataset usando a lib 'datasets'
# O formato JSONL é carregado como 'json'
dataset = load_dataset("json", data_files=path_to_jsonl, split="train")

# 3. Verificação rápida
print(f"✅ Dataset carregado com sucesso! Total de exemplos: {len(dataset)}")
print("Exemplo do primeiro registro:")
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

✅ Dataset carregado com sucesso! Total de exemplos: 16428
Exemplo do primeiro registro:
{'text': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nVocê é um assistente médico especializado do Hospital PosTech. Forneça respostas técnicas, seguras e baseadas nos protocolos internos.\n\n### Input:\nWho is at risk for Lymphocytic Choriomeningitis (LCM)? ?\n\n### Response:\nLCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.</s>'}


In [6]:
# 1. Instalação do motor do Unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. A "Dupla de Ouro" travada nas versões sem o bug do Unpack
!pip install transformers==4.57.1 trl==0.24.0

# 3. Restante das dependências
!pip install peft accelerate bitsandbytes datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-3i6pq4qm/unsloth_f62c6ece301c472b8d3e36bf895e9743
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-3i6pq4qm/unsloth_f62c6ece301c472b8d3e36bf895e9743
  Resolved https://github.com/unslothai/unsloth.git to commit 629199e3a67feba9c06bf94c8bd17dc79948a33a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.6/182.6 kB 15.0 MB/s eta 0:00:00


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled huggingface_hub-1.6.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.2.0
    Uninstalling transformers-5.2.0:
      Successfully uninstalled transformers-5.2.0


In [2]:
import unsloth
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Configurando LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("✅ Modelo pronto para a T4!")

==((====))==  Unsloth 2026.3.4: Fast Mistral patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Modelo pronto para a T4!


In [5]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Garante que o modelo não use cache durante o treino (economiza VRAM na T4)
model.config.use_cache = False

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer, # Com as versões 5.2.0 e 0.24.0, isso não dará mais erro!
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1, # O essencial para a T4
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 110,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/16428 [00:00<?, ? examples/s]

In [6]:

print("Iniciando o treinamento do Mistral...")
trainer_stats = trainer.train()

Iniciando o treinamento do Mistral...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16,428 | Num Epochs = 1 | Total steps = 110
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
1,1.835400
2,1.399400
3,1.587700
4,1.363600
5,1.187300
6,1.120200
7,1.053900
8,0.950000
9,0.776900
10,0.794600


wandb: WARNING URL not available in offline run


train/epoch,▁▁▁▁▂▂▂▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇██
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇████
train/grad_norm,█▃▃▃▂▂▂▂▁▂▂▁▂▂▂▁▂▁▂▁▁▁▂▁▁▁▁▁▂▁▂▂▁▁▁▃▁▁▁▂
train/learning_rate,▁▂▄▅███▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▁
train/loss,█▇▅▃▄▂▄▃▃▃▄▃▄▃▂▃▂▃▄▂▂▂▄▂▂▃▂▁▄▂▄▃▂▂▂▃▄▃▄▄
total_flos,1.4401431179722752e+16
train/epoch,0.05357
train/global_step,110
train/grad_norm,0.70476
train/learning_rate,0.0
train/loss,0.543


In [7]:
import shutil
import os

# Define o caminho exato lá dentro do seu Drive
caminho_drive = "/content/drive/MyDrive/mistral_lora_postech"

# Salva o modelo e o tokenizer direto na nuvem
model.save_pretrained(caminho_drive)
tokenizer.save_pretrained(caminho_drive)

print(f"✅ Pesos LoRA salvos com sucesso direto no Drive em: {caminho_drive}")

✅ Pesos LoRA salvos com sucesso direto no Drive em: /content/drive/MyDrive/mistral_lora_postech
